# Figure 2 — Hit intensity versus time of flightPanel a: ionized water (H2O+) and its fragment (OH+), with the Si+ featuredispersing to higher kinetic energy and the slope change at the warm densematter transition.Panel b: the same intensity axis for H+, H2+ and H3+.Outputs: `figures/data/fig2a_*.npy`, `fig2b_*.npy`.

> **Data.** These notebooks read the preprocessed per-run HDF5 files produced by> `scripts/preprocess.py`. Point `EUXFEL_NANO_DATA` at the directory holding> them. The raw data is at <https://doi.org/10.22003/XFEL.EU-DATA-003408-00>,> under embargo until publication.>> Figures 1-3 can be rebuilt **without** this data from the arrays already in> `figures/data/` — see `figures/README.md`.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import gridspec, cm
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from scipy import constants
from scipy.optimize import curve_fit

from euxfel_nano import config
from euxfel_nano.constants import TIME_BETWEEN_PULSES, CHANNELS_PER_PULSE, channel_time
from euxfel_nano.io import read, read_ion, read_pnccd, data_root
from euxfel_nano.selection import (ion_selection, select_right_of_line,
                                   fish_plot_ion_selection)
from euxfel_nano.calibration import apply_calibration, mq_of, tof_of
from euxfel_nano.fields import build_charge_map, build_charge_maps
from euxfel_nano.yields import (FieldBinnedData, get_yields, significance,
                                verify_errors, sat_func, dec_func,
                                hist_for_selection)

P = config.params()
print("data root:", data_root())

## SelectionOnly events to the right of the dividing line are used, separating the primaryemission mechanism from the argon channel treated in a separate publication(SI "Separation of Reaction Mechanisms", Fig. S5).

In [ ]:
RUNID500 = config.run_ids("500")
THRESHOLD = [(500, 5000)]
ion_dfevent500, ion_dfpulse500 = ion_selection(RUNID500, THRESHOLD)[0]
selected_etof = xr.open_dataarray("selected_etof500.nc")

divider = P["spatial"]["divider_line"]
(outside_spatial_selected_dfevent500,
 outside_spatial_selected_dfpulse500,
 outside_spatial_selected_etof500) = select_right_of_line(
    ion_dfevent500, ion_dfpulse500, selected_etof, divider[0], divider[1])

## Panel b — H+, H2+, H3+

In [ ]:
NBINS_TOF = 4000
NBINS_EVENTS = 60
# NBINS_EVENTS = 140

filtered_dfevents500, filtered_dfpulses500, hists500detail, binsdetail = nions_binning_tof(current_dfevent,current_dfpulse,NBINS_EVENTS,NBINS_TOF)

hists = hists500detail
nbins_tof = NBINS_TOF
xlimits = (1e-7,5e-7)
# xlimits = (1.08e-6,1.39e-6)

precision = TIME_BETWEEN_PULSES/nbins_tof
x_lower, x_upper = xlimits
hist_lower, hist_upper = int(x_lower/precision), int(x_upper/precision)
nbins_events = len(hists)
hists_norm = []

for i in range(nbins_events):

    shortened_hist = hists[i][hist_lower:hist_upper]
    hist_norm = shortened_hist / max(shortened_hist)
    hists_norm.append(hist_norm)

x_edges = np.linspace(x_lower, x_upper, hist_upper-hist_lower)
X, Y = np.meshgrid(x_edges, binsdetail[:-1])


hists_norm_h = hists_norm
Y_h = Y
X_ns = X * 1e9

# hists_norm_si = hists_norm
# Y_si = Y
# X_us = X * 1e6

In [ ]:
plt.figure(figsize=(20, 8))
c = plt.pcolormesh(X_ns, Y_h, hists_norm_h, cmap='magma', shading='auto')
colorbar = plt.colorbar(c, label='Normalized counts (arb. units)', extend='max')
plt.xlabel('Time of flight (ns)', fontsize=20)
plt.ylabel('Hit intensity (arb. unit)', fontsize=20)
plt.xlim(120, 490)
plt.ylim(501,3900)

ax1 = plt.gca()
ax1.tick_params(axis='x', labelsize=16)
ax1.tick_params(axis='y', labelsize=16)
colorbar.ax.set_ylabel('Normalized counts (arb. unit)', fontsize=20)
colorbar.ax.tick_params(labelsize=16)

ax2 = ax1.twiny()
top_tick_positions = [209.2,227]
top_tick_labels = ['150','0']
ax2.set_xticks(top_tick_positions)
ax2.set_xticklabels(top_tick_labels)

ax2.set_xlim(ax1.get_xlim())
ax2.set_xlabel('Proton Kinetic Energy (eV)', fontsize=20)

ax2.spines['top'].set_color('r')
ax2.xaxis.label.set_color('r')
ax2.tick_params(axis='x', colors='r', labelsize=16)

plt.show()

In [ ]:
np.save("../figures/data/fig2b_X_ns.npy", X_ns)
np.save("../figures/data/fig2b_Y_h.npy", Y_h)
np.save("../figures/data/fig2b_hists_norm_h.npy", hists_norm_h)

## Panel a — H2O+, OH+ and Si+

In [ ]:
current_dfpulse = outside_spatial_selected_dfpulse500[outside_spatial_selected_dfpulse500.nevents_pulse < 4000]
current_dfevent = outside_spatial_selected_dfevent500[outside_spatial_selected_dfevent500.pulseId.isin(current_dfpulse.pulseId)]

In [ ]:
NBINS_TOF = 4000
NBINS_EVENTS = 60

filtered_dfevents500, filtered_dfpulses500, hists500detail, binsdetail = nions_binning_tof(current_dfevent,current_dfpulse,NBINS_EVENTS,NBINS_TOF)

hists = hists500detail
nbins_tof = NBINS_TOF
xlimits = (1.04e-6,1.39e-6)

precision = TIME_BETWEEN_PULSES/nbins_tof
x_lower, x_upper = xlimits
hist_lower, hist_upper = int(x_lower/precision), int(x_upper/precision)
nbins_events = len(hists)
hists_norm = []

for i in range(nbins_events):

    shortened_hist = hists[i][hist_lower:hist_upper]
    hist_norm = shortened_hist / max(shortened_hist)
    hists_norm.append(hist_norm)

x_edges = np.linspace(x_lower, x_upper, hist_upper-hist_lower)
X, Y = np.meshgrid(x_edges, binsdetail[:-1])

hists_norm_h2o = np.array(hists_norm)
Y_h2o = Y
X_us = X * 1e6

In [ ]:
plt.figure(figsize=(20, 8))
c = plt.pcolormesh(X_us, Y_h2o, hists_norm_h2o, cmap='magma', shading='auto')
colorbar = plt.colorbar(c, label='Normalized counts (arb. units)', extend='max')
plt.xlabel(r'Time of flight ($\mu$s)', fontsize=20)
plt.ylabel('Hit intensity (arb. unit)', fontsize=20)
plt.ylim(501,3900)

ax1 = plt.gca()
ax1.tick_params(axis='x', labelsize=16)
ax1.tick_params(axis='y', labelsize=16)
colorbar.ax.set_ylabel('Normalized counts (arb. unit)', fontsize=20)
colorbar.ax.tick_params(labelsize=16)

plt.show()

In [ ]:
np.save("../figures/data/fig2a_X_us.npy", X_us)
np.save("../figures/data/fig2a_Y_h2o.npy", Y_h2o)
np.save("../figures/data/fig2a_hists_norm_h2o.npy", hists_norm_h2o)